In [1]:
import sys, os
sys.path.append("../")

import jax
jax.config.update("jax_enable_x64", True)

from qd_solve import *
from qd_solve.eig import *
from qd_solve.operator import *
from qd_solve.split import *
from qd_solve.system import *
from qd_solve.timestepper import *
from qd_solve.spaces.pseudospectral import *
from qd_solve.spaces.finite_difference import *
from qd_solve.exp import *

from miscutils.plot import animate

import jax.numpy as jnp
import diffrax

%matplotlib inline
import matplotlib.pyplot as plt
from IPython.display import Image

ModuleNotFoundError: No module named 'qd_solve'

In [ ]:
potential = lambda x: 0.5 * (x @ x)

x0 = (-10.0, -10.0)
xf = (10.0, 10.0)
num_steps = (500, 500)
num_modes = 250

hilbert_space = FiniteDifference(x0, xf, num_steps)
X, Y = hilbert_space.x_meshgrid

V = FiniteDifferencePotentialEnergy(potential)
L = FiniteDifferenceLaplacian()
#L = L.set_exponentiator(ScaleSquareExponentiator(L, hilbert_space, 0.01))
H = -0.5 * L + V

In [ ]:
x0 = jnp.array(5)
y_fn = lambda x: jnp.exp(-0.5 * (x - x0) @ (x - x0))
y_vals = hilbert_space.eval(y_fn)
y = hilbert_space.from_values(y_vals)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(6, 5), dpi=150)

ax.pcolormesh(X, Y, jnp.abs(y.values), cmap="Blues")

plt.show()
plt.close(fig)

In [ ]:
t0, t1 = 0., 4 * jnp.pi
dt = 0.01

sys = TimeInvariantSystem(H)
%time ys, t_range = sys.propagate(t0, t1, dt, y, Midpoint())

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5), dpi=150)

#ax.plot(hilbert_space.x_ranges[0], jnp.real(ys[100].values))

ax.pcolormesh(X, Y, jnp.abs(ys[0].values), cmap="Blues")

plt.show()
plt.close(fig)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5), dpi=150)

ax.pcolormesh(X, Y, jnp.abs(ys[-1].values), cmap="Blues")

plt.show()
plt.close(fig)

In [ ]:
def func(idx, ax, state, args):
    y_vals = ys[idx].values
    ax[0, 0].pcolormesh(X, Y, jnp.real(y_vals),  vmin=0, vmax=1.0, cmap="Blues")
    ax[0, 1].pcolormesh(X, Y, jnp.imag(y_vals),  vmin=0, vmax=1.0, cmap="Blues")
    ax[1, 0].pcolormesh(X, Y, jnp.abs(y_vals),   vmin=0, vmax=1.0, cmap="Blues")
    ax[1, 1].pcolormesh(X, Y, jnp.angle(y_vals), vmin=-jnp.pi, vmax=jnp.pi, cmap="hsv")
    
%time animate(func, t_range.shape[0] // 2, filename="output.mp4", shape=(2, 2), figsize=(6, 3))
#Image(url="output.gif")  